### MultiQueryoRetriever

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [ ]:
# !uv add langchain langchain_openai

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add langchain_teddynote

In [5]:
from langchain_teddynote import logging

logging.langsmith("test0922")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0922


In [ ]:
# !uv add langchain_community

In [ ]:
# !uv add beautifulsoup4

In [ ]:
# !uv add faiss-cpu

In [13]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader(
    "https://teddylee777.github.io/openai/openai-assistant-tutorial/", encoding="utf-8"
)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = loader.load_and_split(text_splitter)

openai_embedding = OpenAIEmbeddings()

db = FAISS.from_documents(docs, openai_embedding)

retriever = db.as_retriever()

query = "OpenAI Assistant API의 Functions 사용법에 대해 알려줘"
relevant_docs = retriever.invoke(query)

len(relevant_docs)

4

In [14]:
print(relevant_docs[1].page_content)

Assistant + tools(도구)
Assistants API의 핵심 기능 중 하나는 Code Interpreter, Retrieval, 그리고 사용자 정의 함수(OpenAI Functions)와 같은 도구로 우리가 만든 Assistants가 이러한 도구들을 활용할 수 있도록 설정할 수 있습니다.
아래의 튜토리얼은 각각의 도구가 가지는 역할과 설정하는 방법에 대해 자세히 알아보도록 하겠습니다.
도구1: Code Interpreter(코드 인터프리터)
개요


Code Interpreter를 사용하면 어시스턴트 API가 샌드박스가 적용된 실행 환경에서 Python 코드를 작성하고 실행할 수 있습니다.


이 도구는 다양한 데이터와 형식의 파일을 처리하고 데이터와 그래프 이미지가 포함된 파일을 생성할 수 있습니다.


코드 인터프리터를 사용하면 어시스턴트가 코드를 반복적으로 실행하여 까다로운 코드 및 수학 문제를 해결할 수 있습니다.


요금


In [15]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm,
)

In [16]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [17]:
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려줘"

relevant_docs = multiquery_retriever.invoke(question)

print(
    f"==========\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n===========\n",
)

print(relevant_docs[0].page_content)

검색된 문서 개수: 4
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용


In [18]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    """You are an AI language model assistant. 
Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. 
By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
Your response should be a list of values separated by new lines, eg: `foo\nbar\nbaz\n`

#ORIGINAL QUESTION: 
{question}

#Answer in Korean:
"""
)

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

custom_multiquery_chain = (
    {"question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

question = "OpenAI Assistant API의 Functions 사용법에 대해 알려줘"

multi_queries = custom_multiquery_chain.invoke(question)

multi_queries

'OpenAI Assistant API의 Functions 기능을 사용하는 방법에 대해 설명해 주세요  \nOpenAI Assistant API에서 Functions를 활용하는 방법이 궁금합니다  \nOpenAI Assistant API의 Functions를 어떻게 사용할 수 있는지 알려주세요  \nOpenAI Assistant API의 Functions 사용법에 대한 자세한 정보를 제공해 주세요  \nOpenAI Assistant API의 Functions 기능에 대한 사용 지침을 알고 싶습니다  '

In [19]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    llm=custom_multiquery_chain, retriever=db.as_retriever()
)

In [20]:
relevant_docs = multiquery_retriever.invoke(question)

print(
    f"============\n검색된 문서 개수: {len(relevant_docs)}",
    end="\n=================\n",
)

print(relevant_docs[0].page_content)

검색된 문서 개수: 5
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용
